# 从零复现 Faster R-CNN：anchors、RPN、近似 ROI Pooling 与两阶段检测

本 Notebook 不调用 `torchvision` 检测器、`torchvision.ops`、预训练 backbone 或现成 NMS/ROI Align。我们手写 Tiny backbone、anchor 生成、IoU 匹配与均衡采样、box delta 编解码、RPN、proposal 过滤、NMS、整数边界 ROI pooling、二阶段分类/回归头、训练损失和推理后处理。

这里的类名明确使用 `IntegerROIPool`，因为它采用整数裁剪加 adaptive max pooling，**不是**带双线性采样的 ROI Align。全部离线、CPU 单线程；小方块任务只验证两阶段数据流和梯度，不追求或声称真实 mAP。


## 1. 两阶段计算图与坐标合同

```text
images + padding_mask
  -> TinyBackbone -> feature + conservative feature padding mask
  -> RPN conv -> objectness [B,A*Hf*Wf] + deltas [B,A*Hf*Wf,4]
  -> decode -> clip -> size filter -> top-k -> NMS -> proposals
  -> IntegerROIPool(feature, proposals) -> [R,C,3,3]
  -> TwoStageHead -> class logits [R,K+1] + class-agnostic deltas [R,4]
  -> train: RPN loss + ROI classification/regression loss
  -> infer: decode -> per-class NMS -> detections
```

本册统一采用连续边界 `xyxy=(x1,y1,x2,y2)`，范围为 `0<=x<=W, 0<=y<=H`，宽高分别是 `x2-x1,y2-y1`；类别 `0` 专用于 background，真实类别从 1 开始。把 inclusive pixel 坐标与连续边界混用，会产生系统性的 `+1/-1` 误差。


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from copy import deepcopy
from dataclasses import dataclass
from hashlib import sha256
from types import MappingProxyType
import io
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 470728
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})


## 2. Box、IoU 与 delta：先锁住几何语义

给 anchor $a=(x_a,y_a,w_a,h_a)$ 和目标 $g$，常用参数化是

$$t_x=(x_g-x_a)/w_a,\;t_y=(y_g-y_a)/h_a,\;t_w=\log(w_g/w_a),\;t_h=\log(h_g/h_a).$$

decode 是它的逆变换。宽高的指数必须 clamp，否则异常 delta 会溢出。下面不仅测 shape，还验证 encode/decode 往返和手算 IoU；退化 box、NaN、越界 GT 都会 fail closed。


In [ ]:
def validate_boxes(boxes, image_size=None, allow_empty=True):
    if boxes.ndim != 2 or boxes.shape[-1] != 4 or not torch.is_floating_point(boxes):
        raise ValueError("boxes must be floating [N,4]")
    if not torch.isfinite(boxes).all():
        raise ValueError("boxes must be finite")
    if boxes.numel() == 0:
        if allow_empty:
            return
        raise ValueError("boxes cannot be empty")
    if not ((boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])).all():
        raise ValueError("boxes must have positive extent")
    if image_size is not None:
        h, w = image_size
        if (boxes[:, 0] < 0).any() or (boxes[:, 1] < 0).any() or (boxes[:, 2] > w).any() or (boxes[:, 3] > h).any():
            raise ValueError("boxes exceed continuous image bounds")

def box_iou(boxes1, boxes2):
    validate_boxes(boxes1)
    validate_boxes(boxes2)
    if boxes1.shape[0] == 0 or boxes2.shape[0] == 0:
        return boxes1.new_zeros((boxes1.shape[0], boxes2.shape[0]))
    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    intersection = (bottom_right - top_left).clamp(min=0).prod(-1)
    area1 = (boxes1[:, 2:] - boxes1[:, :2]).prod(-1)
    area2 = (boxes2[:, 2:] - boxes2[:, :2]).prod(-1)
    return intersection / (area1[:, None] + area2[None, :] - intersection).clamp(min=1e-8)

class BoxCoder(nn.Module):
    def __init__(self, max_log_scale=math.log(1000.0 / 16)):
        super().__init__()
        self.max_log_scale = float(max_log_scale)

    def encode(self, anchors, targets):
        validate_boxes(anchors); validate_boxes(targets)
        if anchors.shape != targets.shape:
            raise ValueError("anchors and targets must align")
        ac = (anchors[:, :2] + anchors[:, 2:]) / 2
        awh = anchors[:, 2:] - anchors[:, :2]
        gc = (targets[:, :2] + targets[:, 2:]) / 2
        gwh = targets[:, 2:] - targets[:, :2]
        return torch.cat([(gc - ac) / awh, torch.log(gwh / awh)], dim=-1)

    def decode(self, anchors, deltas):
        validate_boxes(anchors)
        if deltas.shape != anchors.shape or not torch.isfinite(deltas).all():
            raise ValueError("deltas must be finite and align with anchors")
        ac = (anchors[:, :2] + anchors[:, 2:]) / 2
        awh = anchors[:, 2:] - anchors[:, :2]
        center = ac + deltas[:, :2] * awh
        wh = awh * deltas[:, 2:].clamp(max=self.max_log_scale).exp()
        return torch.cat([center - wh / 2, center + wh / 2], dim=-1)

    def forward(self, anchors, deltas):
        return self.decode(anchors, deltas)

def clip_boxes(boxes, image_size):
    if boxes.ndim != 2 or boxes.shape[-1] != 4 or not torch.isfinite(boxes).all():
        raise ValueError("finite [N,4] boxes required")
    h, w = image_size
    return torch.stack([boxes[:, 0].clamp(0, w), boxes[:, 1].clamp(0, h),
                        boxes[:, 2].clamp(0, w), boxes[:, 3].clamp(0, h)], dim=-1)

box_coder47 = BoxCoder()
anchors_oracle = torch.tensor([[0., 0., 10., 10.], [10., 8., 20., 24.]])
targets_oracle = torch.tensor([[1., 2., 9., 8.], [8., 10., 24., 22.]])
deltas_oracle = box_coder47.encode(anchors_oracle, targets_oracle)
assert torch.allclose(box_coder47.decode(anchors_oracle, deltas_oracle), targets_oracle, atol=1e-5)
iou_oracle = box_iou(torch.tensor([[0., 0., 2., 2.]]), torch.tensor([[1., 1., 3., 3.]]))
assert torch.allclose(iou_oracle, torch.tensor([[1 / 7]]), atol=1e-7)
assert torch.equal(clip_boxes(torch.tensor([[-2., -1., 35., 40.]]), (32, 32)), torch.tensor([[0., 0., 32., 32.]]))
try:
    validate_boxes(torch.tensor([[1., 1., 1., 3.]]))
    raise AssertionError("degenerate boxes must fail")
except ValueError:
    pass


## 3. Tiny backbone 与 padding 感受野

仅把输入 padding 像素设零还不够：卷积边界附近的 feature cell 感受野可能跨入 padding。这里逐层用与卷积相同的 kernel/stride/padding 对 invalid mask 做 max pooling；任一输入无效，该 feature cell 就标为无效并清零。这是保守策略，会牺牲少量边界特征，但保证调用方任意修改 padding 值都不影响模型输出。


In [ ]:
class TinyDetectionBackbone(nn.Module):
    def __init__(self, in_channels=1, out_channels=16):
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.conv1 = nn.Conv2d(in_channels, 12, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(12, out_channels, 3, stride=2, padding=1)

    @staticmethod
    def _advance_mask(mask):
        return F.max_pool2d(mask.float().unsqueeze(1), 3, stride=2, padding=1).squeeze(1).bool()

    def forward(self, images, padding_mask):
        if images.ndim != 4 or images.shape[1] != self.in_channels:
            raise ValueError("images must be configured NCHW")
        if padding_mask.shape != images.shape[:1] + images.shape[-2:] or padding_mask.dtype != torch.bool:
            raise ValueError("padding_mask must be bool [B,H,W]")
        if not torch.isfinite(images).all():
            raise ValueError("images must be finite")
        x = images.masked_fill(padding_mask[:, None], 0)
        mask = self._advance_mask(padding_mask)
        x = F.relu(self.conv1(x)).masked_fill(mask[:, None], 0)
        mask = self._advance_mask(mask)
        x = F.relu(self.conv2(x)).masked_fill(mask[:, None], 0)
        return x, mask

backbone47 = TinyDetectionBackbone()
pad_mask_probe = torch.zeros(1, 32, 32, dtype=torch.bool)
pad_mask_probe[:, 24:, :] = True; pad_mask_probe[:, :, 24:] = True
pad_a = torch.randn(1, 1, 32, 32)
pad_b = pad_a.clone(); pad_b[pad_mask_probe[:, None]] = 999.0
feat_a, fmask_a = backbone47(pad_a, pad_mask_probe)
feat_b, fmask_b = backbone47(pad_b, pad_mask_probe)
assert feat_a.shape == (1, 16, 8, 8)
assert torch.equal(fmask_a, fmask_b)
assert torch.equal(feat_a, feat_b)
assert torch.equal(feat_a.masked_select(fmask_a[:, None]), torch.zeros_like(feat_a.masked_select(fmask_a[:, None])))


## 4. Anchors、IoU 匹配与均衡采样

每个 feature cell 以其中心为 anchor 中心，并枚举三个 aspect ratio。RPN label 为 `1=positive, 0=negative, -1=ignore`：IoU 高于正阈值为正、低于负阈值为负，中间忽略；另外强制每个 GT 的最佳 anchor 为正，避免小物体没有监督。空 GT 图像的所有 anchor 都是负样本，这是目标检测训练必须覆盖的正常情况。


In [ ]:
RPN_POSITIVE_IOU47 = 0.5
RPN_NEGATIVE_IOU47 = 0.2
RPN_BATCH_SIZE47 = 64
RPN_POSITIVE_FRACTION47 = 0.5

class AnchorGenerator(nn.Module):
    def __init__(self, size=8.0, aspect_ratios=(0.5, 1.0, 2.0)):
        super().__init__()
        if size <= 0 or not aspect_ratios or any(r <= 0 for r in aspect_ratios):
            raise ValueError("invalid anchor recipe")
        self.size = float(size)
        self.aspect_ratios = tuple(float(r) for r in aspect_ratios)

    def forward(self, feature_size, image_size):
        hf, wf = feature_size; hi, wi = image_size
        if min(hf, wf, hi, wi) <= 0:
            raise ValueError("feature/image dimensions must be positive")
        stride_y, stride_x = hi / hf, wi / wf
        cy = (torch.arange(hf, dtype=torch.float32) + 0.5) * stride_y
        cx = (torch.arange(wf, dtype=torch.float32) + 0.5) * stride_x
        yy, xx = torch.meshgrid(cy, cx, indexing="ij")
        all_anchors = []
        for ratio in self.aspect_ratios:
            width = self.size * math.sqrt(ratio)
            height = self.size / math.sqrt(ratio)
            all_anchors.append(torch.stack([xx - width/2, yy - height/2, xx + width/2, yy + height/2], -1))
        return torch.stack(all_anchors, 2).reshape(-1, 4)

def match_anchors(anchors, gt_boxes, positive_iou=RPN_POSITIVE_IOU47, negative_iou=RPN_NEGATIVE_IOU47):
    validate_boxes(anchors); validate_boxes(gt_boxes)
    if not 0 <= negative_iou < positive_iou <= 1:
        raise ValueError("invalid IoU thresholds")
    labels = torch.full((anchors.shape[0],), -1, dtype=torch.long)
    matched = torch.zeros(anchors.shape[0], dtype=torch.long)
    if gt_boxes.shape[0] == 0:
        labels.fill_(0)
        return labels, matched
    ious = box_iou(anchors, gt_boxes)
    best_iou, matched = ious.max(dim=1)
    labels[best_iou < negative_iou] = 0
    labels[best_iou >= positive_iou] = 1
    best_anchor_per_gt = ious.argmax(dim=0)
    labels[best_anchor_per_gt] = 1
    matched[best_anchor_per_gt] = torch.arange(gt_boxes.shape[0])
    return labels, matched

def balanced_sample(labels, batch_size, positive_fraction, generator):
    if labels.ndim != 1 or labels.dtype != torch.long or not set(labels.unique().tolist()).issubset({-1, 0, 1}):
        raise ValueError("labels must use {-1,0,1}")
    if not isinstance(batch_size, int) or batch_size <= 0 or not 0 <= positive_fraction <= 1:
        raise ValueError("sampling batch_size/fraction contract is invalid")
    if not isinstance(generator, torch.Generator):
        raise ValueError("balanced sampling requires an explicit torch.Generator")
    positive = torch.where(labels == 1)[0]
    negative = torch.where(labels == 0)[0]
    npos = min(int(batch_size * positive_fraction), positive.numel())
    nneg = min(batch_size - npos, negative.numel())
    pos = positive[torch.randperm(positive.numel(), generator=generator)[:npos]]
    neg = negative[torch.randperm(negative.numel(), generator=generator)[:nneg]]
    return torch.cat([pos, neg]), pos

anchor_generator47 = AnchorGenerator()
anchors47 = anchor_generator47((8, 8), (32, 32))
assert anchors47.shape == (8 * 8 * 3, 4)
empty_labels, empty_match = match_anchors(anchors47, torch.empty(0, 4))
assert torch.equal(empty_labels, torch.zeros_like(empty_labels))
assert empty_match.shape == empty_labels.shape
one_labels, one_match = match_anchors(anchors47, torch.tensor([[8., 8., 16., 16.]]))
assert bool((one_labels == 1).any()) and bool((one_labels == 0).any()) and bool((one_labels == -1).any())
sampled47, sampled_pos47 = balanced_sample(one_labels, 32, 0.5, torch.Generator().manual_seed(4))
sampled_again47, _ = balanced_sample(one_labels, 32, 0.5, torch.Generator().manual_seed(4))
assert sampled47.numel() <= 32 and sampled_pos47.numel() <= 16
assert sampled47.unique().numel() == sampled47.numel() and not bool((one_labels[sampled47] == -1).any())
assert torch.equal(sampled47, sampled_again47)


## 5. RPN：objectness 与 class-agnostic box regression

共享 `3×3` 卷积后，每个位置、每个 anchor 输出一个 objectness logit 和四个 delta。分类损失只看 sampled positive/negative；回归损失只看 positive。禁止对 ignore anchor 算 BCE，也不能在空 GT 时对空回归张量求 mean（会得到 NaN）。


In [ ]:
RPN_SMOOTH_L1_BETA47 = 1 / 9

class RegionProposalNetwork(nn.Module):
    def __init__(self, in_channels, anchors_per_location=3):
        super().__init__()
        if in_channels <= 0 or anchors_per_location <= 0:
            raise ValueError("RPN dimensions must be positive")
        self.anchors_per_location = int(anchors_per_location)
        self.shared = nn.Conv2d(in_channels, in_channels, 3, padding=1)
        self.objectness = nn.Conv2d(in_channels, anchors_per_location, 1)
        self.box_deltas = nn.Conv2d(in_channels, 4 * anchors_per_location, 1)

    def forward(self, features):
        if features.ndim != 4 or not torch.is_floating_point(features) or not torch.isfinite(features).all():
            raise ValueError("RPN expects finite floating NCHW features")
        hidden = F.relu(self.shared(features))
        logits = self.objectness(hidden).permute(0, 2, 3, 1).reshape(features.shape[0], -1)
        deltas = self.box_deltas(hidden).reshape(features.shape[0], self.anchors_per_location, 4,
                                                       features.shape[2], features.shape[3])
        deltas = deltas.permute(0, 3, 4, 1, 2).reshape(features.shape[0], -1, 4)
        return logits, deltas

def rpn_losses(logits, deltas, anchors, targets, generator, anchor_valid=None):
    if (logits.ndim != 2 or deltas.ndim != 3 or logits.shape[:2] != deltas.shape[:2]
            or deltas.shape[-1] != 4 or logits.shape[1] != anchors.shape[0]):
        raise ValueError("RPN prediction/anchor shape mismatch")
    if not torch.is_floating_point(logits) or not torch.is_floating_point(deltas):
        raise ValueError("RPN predictions must be floating point")
    if not torch.isfinite(logits).all() or not torch.isfinite(deltas).all():
        raise ValueError("RPN predictions must be finite")
    validate_boxes(anchors)
    if len(targets) != logits.shape[0]:
        raise ValueError("one RPN target per image is required")
    cls_losses, reg_losses = [], []
    if anchor_valid is None:
        anchor_valid = torch.ones_like(logits, dtype=torch.bool)
    if anchor_valid.shape != logits.shape or anchor_valid.dtype != torch.bool:
        raise ValueError("anchor_valid must be bool and align with objectness")
    for batch_index, target in enumerate(targets):
        if set(target) != {"boxes", "labels"}:
            raise ValueError("RPN target must contain boxes and labels")
        labels, matched = match_anchors(anchors, target["boxes"])
        labels[~anchor_valid[batch_index]] = -1
        # 全 padding 且空 GT 是合法占位样本，返回与预测图相连的可导零。
        if not bool(anchor_valid[batch_index].any()):
            if target["boxes"].numel():
                raise ValueError("fully padded image cannot carry ground truth")
            cls_losses.append(logits[batch_index].sum() * 0)
            reg_losses.append(deltas[batch_index].sum() * 0)
            continue
        sampled, positives = balanced_sample(labels, RPN_BATCH_SIZE47, RPN_POSITIVE_FRACTION47, generator)
        if sampled.numel():
            cls_sum = F.binary_cross_entropy_with_logits(
                logits[batch_index, sampled], labels[sampled].float(), reduction="sum")
            cls_losses.append(cls_sum / sampled.numel())
        else:
            cls_losses.append(logits[batch_index].sum() * 0)
        if positives.numel():
            wanted = box_coder47.encode(anchors[positives], target["boxes"][matched[positives]])
            reg_sum = F.smooth_l1_loss(deltas[batch_index, positives], wanted,
                                       beta=RPN_SMOOTH_L1_BETA47, reduction="sum")
            reg_losses.append(reg_sum / (positives.numel() * 4))
        else:
            reg_losses.append(deltas[batch_index].sum() * 0)
    return torch.stack(cls_losses).mean(), torch.stack(reg_losses).mean()

rpn47 = RegionProposalNetwork(16)
rpn_logits_probe, rpn_deltas_probe = rpn47(torch.randn(2, 16, 8, 8))
assert rpn_logits_probe.shape == (2, 192)
assert rpn_deltas_probe.shape == (2, 192, 4)
assert torch.isfinite(rpn_logits_probe).all() and torch.isfinite(rpn_deltas_probe).all()

all_invalid_logits47 = torch.zeros(1, anchors47.shape[0], requires_grad=True)
all_invalid_deltas47 = torch.zeros(1, anchors47.shape[0], 4, requires_grad=True)
empty_rpn_target47 = [{"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}]
zero_rpn_cls47, zero_rpn_reg47 = rpn_losses(
    all_invalid_logits47, all_invalid_deltas47, anchors47, empty_rpn_target47,
    torch.Generator().manual_seed(47), torch.zeros_like(all_invalid_logits47, dtype=torch.bool))
assert torch.equal(zero_rpn_cls47, torch.tensor(0.0)) and torch.equal(zero_rpn_reg47, torch.tensor(0.0))
assert torch.isfinite(zero_rpn_cls47 + zero_rpn_reg47)
(zero_rpn_cls47 + zero_rpn_reg47).backward()
assert all_invalid_logits47.grad is not None and torch.count_nonzero(all_invalid_logits47.grad) == 0


## 6. Proposal：decode、clip、尺寸过滤、top-k 与手写 NMS

NMS 按 score 从高到低保留 box，并删除与当前 box 的 IoU 超过阈值者。顺序很重要：应先 clip/删除退化框，再做 NMS。随机 RPN 可能产生零 proposal，后续 ROI 与推理都必须把空张量当正常输入，而不是假设至少有一个框。


In [ ]:
RPN_PROPOSAL_SCORE47 = 0.0
RPN_PRE_NMS_TOPK47 = 80
RPN_POST_NMS47 = 24
RPN_NMS_IOU47 = 0.7
RPN_MIN_SIZE47 = 1.0

def nms(boxes, scores, iou_threshold):
    validate_boxes(boxes)
    if scores.ndim != 1 or scores.shape[0] != boxes.shape[0] or not torch.is_floating_point(scores) or not torch.isfinite(scores).all():
        raise ValueError("scores must align and be finite floating point")
    if not math.isfinite(iou_threshold) or not 0 <= iou_threshold <= 1:
        raise ValueError("invalid NMS threshold")
    if boxes.shape[0] == 0:
        return torch.empty(0, dtype=torch.long, device=boxes.device)
    order = scores.argsort(descending=True)
    keep = []
    while order.numel():
        current = order[0]
        keep.append(current)
        if order.numel() == 1:
            break
        rest = order[1:]
        order = rest[box_iou(boxes[current:current+1], boxes[rest]).squeeze(0) <= iou_threshold]
    return torch.stack(keep)

def generate_proposals(logits, deltas, anchors, image_size, score_threshold=RPN_PROPOSAL_SCORE47,
                       topk=RPN_PRE_NMS_TOPK47, post_nms=RPN_POST_NMS47):
    if logits.ndim != 1 or not torch.is_floating_point(logits) or not torch.isfinite(logits).all():
        raise ValueError("proposal logits must be finite floating [A]")
    if deltas.shape != anchors.shape or logits.shape[0] != anchors.shape[0]:
        raise ValueError("proposal predictions must align with anchors")
    if not math.isfinite(score_threshold) or not 0 <= score_threshold <= 1 or topk <= 0 or post_nms < 0:
        raise ValueError("proposal filtering recipe is invalid")
    boxes = clip_boxes(box_coder47.decode(anchors, deltas), image_size)
    scores = logits.sigmoid()
    wh = boxes[:, 2:] - boxes[:, :2]
    valid = (wh[:, 0] >= RPN_MIN_SIZE47) & (wh[:, 1] >= RPN_MIN_SIZE47) & (scores >= score_threshold)
    boxes, scores = boxes[valid], scores[valid]
    if boxes.shape[0] == 0:
        return boxes, scores
    top = scores.argsort(descending=True)[:topk]
    boxes, scores = boxes[top], scores[top]
    keep = nms(boxes, scores, RPN_NMS_IOU47)[:post_nms]
    return boxes[keep], scores[keep]

nms_boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.], [20., 20., 30., 30.]])
nms_scores = torch.tensor([0.9, 0.8, 0.7])
assert torch.equal(nms(nms_boxes, nms_scores, 0.5), torch.tensor([0, 2]))
assert nms(torch.empty(0, 4), torch.empty(0), 0.5).numel() == 0
none_boxes, none_scores = generate_proposals(torch.full((192,), -20.), torch.zeros(192, 4),
                                              anchors47, (32, 32), score_threshold=0.99)
assert none_boxes.shape == (0, 4) and none_scores.shape == (0,)


## 7. `IntegerROIPool`：它不是 ROI Align

本实现把 image 坐标乘 `feature/image` scale，左上取 floor、右下取 ceil，裁剪整数 feature 区域后做 `adaptive_max_pool2d`。这相当于经典 ROI pooling 的简化版，会有量化误差；ROI Align 则在浮点采样点做双线性插值并避免两次取整。为了不误导，类名和接口都明确写出 `Integer`。


In [ ]:
class IntegerROIPool(nn.Module):
    def __init__(self, output_size=(3, 3)):
        super().__init__()
        self.output_size = tuple(output_size)

    def forward(self, features, rois, image_size):
        if features.ndim != 4 or rois.ndim != 2 or rois.shape[1] != 5:
            raise ValueError("features NCHW and rois [R,batch,x1,y1,x2,y2] required")
        if not torch.isfinite(rois).all():
            raise ValueError("rois must be finite")
        if rois.shape[0] == 0:
            return features.new_empty((0, features.shape[1], *self.output_size))
        validate_boxes(rois[:, 1:], image_size)
        hi, wi = image_size; hf, wf = features.shape[-2:]
        pooled = []
        for roi in rois:
            batch_index = int(roi[0].item())
            if batch_index < 0 or batch_index >= features.shape[0] or float(roi[0]) != batch_index:
                raise ValueError("ROI batch index is invalid")
            x1 = max(0, min(wf - 1, math.floor(float(roi[1]) * wf / wi)))
            y1 = max(0, min(hf - 1, math.floor(float(roi[2]) * hf / hi)))
            x2 = max(x1 + 1, min(wf, math.ceil(float(roi[3]) * wf / wi)))
            y2 = max(y1 + 1, min(hf, math.ceil(float(roi[4]) * hf / hi)))
            pooled.append(F.adaptive_max_pool2d(features[batch_index:batch_index+1, :, y1:y2, x1:x2], self.output_size)[0])
        return torch.stack(pooled)

roi_pool47 = IntegerROIPool((2, 2))
grid_feature = torch.arange(64, dtype=torch.float32).reshape(1, 1, 8, 8)
grid_roi = torch.tensor([[0., 8., 8., 16., 16.]])
grid_pooled = roi_pool47(grid_feature, grid_roi, (32, 32))
assert torch.equal(grid_pooled[0, 0], torch.tensor([[18., 19.], [26., 27.]]))
assert roi_pool47(torch.randn(2, 4, 8, 8), torch.empty(0, 5), (32, 32)).shape == (0, 4, 2, 2)


## 8. 二阶段 head 与整网 forward

每个 proposal 的 pooled feature 进入两层 MLP，输出 `K+1` 类和一组 class-agnostic delta。训练时把 GT box 追加到 RPN proposals，确保早期至少存在正 ROI；这只是训练采样策略，推理不能偷看 GT。空 GT 图像仍可贡献 background 分类损失。


In [ ]:
class TwoStageHead(nn.Module):
    def __init__(self, in_channels=16, pool_size=3, num_classes=2, hidden=48):
        super().__init__()
        self.num_classes = int(num_classes)
        self.fc1 = nn.Linear(in_channels * pool_size * pool_size, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.classifier = nn.Linear(hidden, num_classes + 1)
        self.regressor = nn.Linear(hidden, 4)

    def forward(self, pooled):
        if pooled.ndim != 4:
            raise ValueError("pooled features must be RCHW")
        if pooled.shape[0] == 0:
            return pooled.new_empty((0, self.num_classes + 1)), pooled.new_empty((0, 4))
        hidden = F.relu(self.fc1(pooled.flatten(1)))
        hidden = F.relu(self.fc2(hidden))
        return self.classifier(hidden), self.regressor(hidden)

def validate_targets47(targets, batch_size, image_size, num_classes):
    if len(targets) != batch_size:
        raise ValueError("one target dictionary per image is required")
    for target in targets:
        if set(target) != {"boxes", "labels"}:
            raise ValueError("target must contain exactly boxes and labels")
        validate_boxes(target["boxes"], image_size)
        labels = target["labels"]
        if labels.ndim != 1 or labels.dtype != torch.long or labels.shape[0] != target["boxes"].shape[0]:
            raise ValueError("labels must be int64 and align with boxes")
        if labels.numel() and ((labels < 1).any() or (labels > num_classes).any()):
            raise ValueError("foreground labels must be in [1,num_classes]")

class TinyFasterRCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.num_classes = int(num_classes)
        self.backbone = TinyDetectionBackbone(1, 16)
        self.anchor_generator = AnchorGenerator(8.0, (0.5, 1.0, 2.0))
        self.rpn = RegionProposalNetwork(16, 3)
        self.roi_pool = IntegerROIPool((3, 3))
        self.head = TwoStageHead(16, 3, num_classes)

    def forward(self, images, padding_mask, targets=None, score_threshold=0.0):
        if images.shape[-2:] != (32, 32):
            raise ValueError("published teaching model expects 32x32 images")
        if targets is not None:
            validate_targets47(targets, images.shape[0], (32, 32), self.num_classes)
        features, feature_padding = self.backbone(images, padding_mask)
        fully_invalid = feature_padding.flatten(1).all(1)
        if targets is not None:
            for batch_index, invalid in enumerate(fully_invalid.tolist()):
                if invalid and targets[batch_index]["boxes"].numel():
                    raise ValueError("fully padded image cannot carry ground truth")
        anchors = self.anchor_generator(features.shape[-2:], images.shape[-2:]).to(features.device)
        objectness, rpn_deltas = self.rpn(features)
        anchor_valid = (~feature_padding).reshape(images.shape[0], -1, 1).expand(-1, -1, 3).reshape(images.shape[0], -1)
        proposals, proposal_scores = [], []
        for b in range(images.shape[0]):
            # Proposal selection/NMS 是离散路径；与标准 two-stage detector 一样在两阶段间 stop-gradient。
            valid = anchor_valid[b]
            boxes, scores = generate_proposals(objectness[b, valid].detach(), rpn_deltas[b, valid].detach(),
                                                anchors[valid], (32, 32), score_threshold)
            if targets is not None and targets[b]["boxes"].numel():
                boxes = torch.cat([boxes, targets[b]["boxes"]], dim=0)
                scores = torch.cat([scores, torch.ones(targets[b]["boxes"].shape[0], device=scores.device)], dim=0)
            proposals.append(boxes); proposal_scores.append(scores)
        roi_rows = [torch.cat([torch.full((boxes.shape[0], 1), float(b), device=boxes.device), boxes], 1)
                    for b, boxes in enumerate(proposals) if boxes.shape[0]]
        rois = torch.cat(roi_rows, 0) if roi_rows else features.new_empty((0, 5))
        pooled = self.roi_pool(features, rois, (32, 32))
        class_logits, box_deltas = self.head(pooled)
        return {"features": features, "feature_padding": feature_padding, "anchors": anchors,
                "objectness": objectness, "rpn_deltas": rpn_deltas, "anchor_valid": anchor_valid, "proposals": proposals,
                "proposal_scores": proposal_scores, "class_logits": class_logits, "box_deltas": box_deltas}

detector47 = TinyFasterRCNN(2)
try:
    validate_targets47([{"boxes": torch.tensor([[2., 2., 8., 8.]]), "labels": torch.tensor([0])}], 1, (32, 32), 2)
    raise AssertionError("background cannot be supplied as GT")
except ValueError:
    pass


## 9. Detection loss：proposal/target 对齐不能靠数组下标

RPN 与 ROI head 各自重新做 IoU matching。ROI 的 background label 为 0；正 proposal 的回归目标由其 matched GT 编码。若一张图没有 ROI，则跳过该图的 ROI loss，同时保留图中其它 batch 成员的监督。所有 loss 都保持为 tensor，不能用 `.item()` 后再拼回计算图。


In [ ]:
ROI_POSITIVE_IOU47 = 0.4
ROI_SMOOTH_L1_BETA47 = 1 / 9
ROI_SAMPLER47 = "all-post-nms-proposals-plus-gt"

def detection_losses47(outputs, targets, generator):
    rpn_cls, rpn_reg = rpn_losses(outputs["objectness"], outputs["rpn_deltas"], outputs["anchors"], targets,
                                  generator, outputs["anchor_valid"])
    roi_cls_losses, roi_reg_losses = [], []
    offset = 0
    for boxes, target in zip(outputs["proposals"], targets):
        count = boxes.shape[0]
        logits = outputs["class_logits"][offset:offset+count]
        deltas = outputs["box_deltas"][offset:offset+count]
        offset += count
        if count == 0:
            continue
        if target["boxes"].shape[0] == 0:
            labels = torch.zeros(count, dtype=torch.long, device=logits.device)
            matched = torch.zeros(count, dtype=torch.long, device=logits.device)
            positive = torch.zeros(count, dtype=torch.bool, device=logits.device)
        else:
            ious = box_iou(boxes, target["boxes"])
            best_iou, matched = ious.max(1)
            positive = best_iou >= ROI_POSITIVE_IOU47
            best_proposal = ious.argmax(0)
            positive[best_proposal] = True
            matched[best_proposal] = torch.arange(target["boxes"].shape[0])
            labels = torch.zeros(count, dtype=torch.long, device=logits.device)
            labels[positive] = target["labels"][matched[positive]]
        cls_sum = F.cross_entropy(logits, labels, reduction="sum")
        roi_cls_losses.append(cls_sum / count)
        if positive.any():
            wanted = box_coder47.encode(boxes[positive], target["boxes"][matched[positive]])
            reg_sum = F.smooth_l1_loss(deltas[positive], wanted, beta=ROI_SMOOTH_L1_BETA47, reduction="sum")
            roi_reg_losses.append(reg_sum / (int(positive.sum()) * 4))
        else:
            roi_reg_losses.append(deltas.sum() * 0)
    if offset != outputs["class_logits"].shape[0]:
        raise RuntimeError("proposal/head offset mismatch")
    zero = outputs["objectness"].sum() * 0
    roi_cls = torch.stack(roi_cls_losses).mean() if roi_cls_losses else zero
    roi_reg = torch.stack(roi_reg_losses).mean() if roi_reg_losses else zero
    return {"rpn_objectness": rpn_cls, "rpn_box": rpn_reg, "roi_class": roi_cls, "roi_box": roi_reg}

def make_detection_batch47():
    images = torch.zeros(3, 1, 32, 32)
    images[0, :, 6:14, 5:13] = 1.0
    images[1, :, 17:27, 18:28] = 0.8
    images[2] = 0.03
    padding = torch.zeros(3, 32, 32, dtype=torch.bool)
    padding[1, 30:, :] = True; padding[1, :, 30:] = True
    targets = [
        {"boxes": torch.tensor([[5., 6., 13., 14.]]), "labels": torch.tensor([1])},
        {"boxes": torch.tensor([[18., 17., 28., 27.]]), "labels": torch.tensor([2])},
        {"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)},
    ]
    return images, padding, targets

train_images47, train_padding47, train_targets47 = make_detection_batch47()
probe_outputs47 = detector47(train_images47, train_padding47, train_targets47)
assert not bool(probe_outputs47["anchor_valid"][1].all())
probe_losses47 = detection_losses47(probe_outputs47, train_targets47, torch.Generator().manual_seed(SEED + 1))
probe_total47 = sum(probe_losses47.values())
probe_total47.backward()
assert all(torch.isfinite(value) for value in probe_losses47.values())
assert detector47.backbone.conv1.weight.grad is not None
assert detector47.rpn.objectness.weight.grad is not None
assert detector47.head.classifier.weight.grad is not None
assert float(detector47.head.classifier.weight.grad.abs().sum()) > 0

# 全 padding + 空 GT 是合法占位样本，所有 loss 为可导零；带 GT 则直接拒绝。
padding_only_model47 = TinyFasterRCNN(2)
padding_only_image47 = torch.zeros(1, 1, 32, 32)
padding_only_mask47 = torch.ones(1, 32, 32, dtype=torch.bool)
padding_only_target47 = [{"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}]
padding_only_output47 = padding_only_model47(padding_only_image47, padding_only_mask47, padding_only_target47)
padding_only_losses47 = detection_losses47(
    padding_only_output47, padding_only_target47, torch.Generator().manual_seed(4701))
padding_only_total47 = sum(padding_only_losses47.values())
assert torch.equal(padding_only_total47, torch.tensor(0.0)) and torch.isfinite(padding_only_total47)
padding_only_total47.backward()
assert padding_only_model47.rpn.objectness.weight.grad is not None

try:
    padding_only_model47(padding_only_image47, padding_only_mask47,
                         [{"boxes": torch.tensor([[2., 2., 8., 8.]]), "labels": torch.tensor([1])}])
    raise AssertionError("fully padded image with GT must fail")
except ValueError as exc:
    assert "fully padded" in str(exc)


## 10. 受控微型训练与推理

我们在固定的三张图上短暂优化，观察联合 loss 是否下降；GT 被追加到训练 proposals，因此这是**优化/接线测试**，不是独立测试集。推理时只使用 RPN proposals，class probability 过滤后按类别 NMS。真实检测评估还需要独立 split、AP across IoU thresholds、尺寸分桶和 error analysis。


In [ ]:
TRAIN_STEPS47 = 28
detector47 = TinyFasterRCNN(2)
optimizer47 = torch.optim.Adam(detector47.parameters(), lr=3e-3)
loss_trace47 = []
detector47.train()
for step in range(TRAIN_STEPS47):
    optimizer47.zero_grad(set_to_none=True)
    outputs = detector47(train_images47, train_padding47, train_targets47)
    parts = detection_losses47(outputs, train_targets47, torch.Generator().manual_seed(SEED + 100 + step))
    total = sum(parts.values())
    total.backward()
    torch.nn.utils.clip_grad_norm_(detector47.parameters(), 5.0)
    optimizer47.step()
    loss_trace47.append(float(total.detach()))

assert min(loss_trace47[-5:]) < 0.75 * loss_trace47[0]
assert all(math.isfinite(v) for v in loss_trace47)
print({"initial_joint_loss": round(loss_trace47[0], 4), "best_last5": round(min(loss_trace47[-5:]), 4)})

ROI_SCORE_THRESHOLD47 = 0.2
ROI_NMS_IOU47 = 0.5

def postprocess_detections47(proposals, class_logits, box_deltas, image_size, score_threshold=ROI_SCORE_THRESHOLD47):
    if proposals.ndim != 2 or proposals.shape[-1] != 4 or not torch.is_floating_point(proposals):
        raise ValueError("proposals must be floating [R,4]")
    if class_logits.ndim != 2 or class_logits.shape[0] != proposals.shape[0] or class_logits.shape[1] < 2:
        raise ValueError("class logits must be [R,K+1]")
    if box_deltas.shape != proposals.shape or not torch.is_floating_point(box_deltas):
        raise ValueError("box deltas must be floating [R,4]")
    if not torch.is_floating_point(class_logits):
        raise ValueError("class logits must be floating point")
    if not torch.isfinite(proposals).all() or not torch.isfinite(class_logits).all() or not torch.isfinite(box_deltas).all():
        raise ValueError("postprocess inputs must be finite")
    if proposals.device != class_logits.device or proposals.device != box_deltas.device:
        raise ValueError("postprocess tensors must share a device")
    if not math.isfinite(score_threshold) or not 0 <= score_threshold <= 1:
        raise ValueError("score_threshold must be finite in [0,1]")
    validate_boxes(proposals, image_size)
    if proposals.shape[0] == 0:
        return {"boxes": proposals, "scores": proposals.new_empty(0),
                "labels": torch.empty(0, dtype=torch.long, device=proposals.device)}
    probabilities = class_logits.softmax(-1)
    all_boxes, all_scores, all_labels = [], [], []
    decoded = clip_boxes(box_coder47.decode(proposals, box_deltas), image_size)
    extent = decoded[:, 2:] - decoded[:, :2]
    valid_extent = (extent[:, 0] >= RPN_MIN_SIZE47) & (extent[:, 1] >= RPN_MIN_SIZE47)
    for label in range(1, class_logits.shape[1]):
        scores = probabilities[:, label]
        selected = (scores >= score_threshold) & valid_extent
        if not selected.any():
            continue
        boxes_l, scores_l = decoded[selected], scores[selected]
        keep = nms(boxes_l, scores_l, ROI_NMS_IOU47)
        all_boxes.append(boxes_l[keep]); all_scores.append(scores_l[keep])
        all_labels.append(torch.full((keep.numel(),), label, dtype=torch.long, device=proposals.device))
    if not all_boxes:
        return {"boxes": proposals.new_empty((0, 4)), "scores": proposals.new_empty(0),
                "labels": torch.empty(0, dtype=torch.long, device=proposals.device)}
    boxes, scores, labels = torch.cat(all_boxes), torch.cat(all_scores), torch.cat(all_labels)
    order = scores.argsort(descending=True)
    return {"boxes": boxes[order], "scores": scores[order], "labels": labels[order]}

empty_detection47 = postprocess_detections47(torch.empty(0, 4), torch.empty(0, 3), torch.empty(0, 4), (32, 32))
assert empty_detection47["boxes"].shape == (0, 4)
assert empty_detection47["labels"].dtype == torch.long

bad_postprocess47 = [
    (torch.tensor([[0., 0., 5., 5.]]), torch.tensor([[0., float("nan"), 0.]]), torch.zeros(1, 4), 0.0),
    (torch.tensor([[0., 0., 5., 5.]]), torch.zeros(1, 3), torch.full((1, 4), float("inf")), 0.0),
    (torch.tensor([[0., 0., 5., 5.]]), torch.zeros(1, 3), torch.zeros(1, 4), float("nan")),
]
for bad_boxes47, bad_logits47, bad_deltas47, bad_threshold47 in bad_postprocess47:
    try:
        postprocess_detections47(bad_boxes47, bad_logits47, bad_deltas47, (32, 32), bad_threshold47)
        raise AssertionError("nonfinite detection input must fail")
    except ValueError as exc:
        assert "finite" in str(exc)

detector47.eval()
with torch.no_grad():
    inference_outputs47 = detector47(train_images47[:2], train_padding47[:2], targets=None, score_threshold=0.0)
    first_count47 = inference_outputs47["proposals"][0].shape[0]
    first_detection47 = postprocess_detections47(
        inference_outputs47["proposals"][0], inference_outputs47["class_logits"][:first_count47],
        inference_outputs47["box_deltas"][:first_count47], (32, 32), 0.05)
assert first_detection47["boxes"].ndim == 2 and first_detection47["boxes"].shape[-1] == 4
assert torch.isfinite(first_detection47["scores"]).all()


## 11. 发布边界：坐标、anchor、采样、loss 分母和推理阈值都要绑定

检测模型的 state dict 相同，并不意味着系统语义相同：anchor 顺序与 ratio 定义、`xyxy` 是否 inclusive、padding mask、IoU 阈值、正负采样、loss 分母、top-k/NMS、类别起点任一变化都会破坏输出。本制品把这些 recipe 和受控数据快照一并纳入 package digest；canonical state digest 逐参数绑定 key/dtype/shape/bytes。

外部只读 publisher registry 保存发布时批准的整体 digest。loader 还逐项交叉验证 recipe，并返回携带不可变 `preprocess/labels/coordinates/anchors/training/inference` 的 `PublishedDetector`，避免调用方拿裸模型猜语义。攻击者即使替换 head、改 label map并重算全部内部 hash，仍无法改变 registry 中的期望值。


In [ ]:
def state_digest47(state):
    digest = sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())
        digest.update(json.dumps(list(tensor.shape)).encode()); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

def data_digest47(images, padding, targets):
    digest = sha256()
    for tensor in [images, padding] + [v for target in targets for v in (target["boxes"], target["labels"])]:
        value = tensor.detach().cpu().contiguous()
        digest.update(str(value.dtype).encode()); digest.update(json.dumps(list(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def package_digest47(package):
    payload = {k: package[k] for k in sorted(package) if k != "package_digest"}
    return sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def deep_freeze47(value):
    if isinstance(value, dict): return MappingProxyType({k: deep_freeze47(v) for k, v in value.items()})
    if isinstance(value, list): return tuple(deep_freeze47(v) for v in value)
    return value

CONFIG47 = {"image_size": [32, 32], "in_channels": 1, "backbone_channels": 16, "num_classes": 2,
            "roi_pool": [3, 3], "roi_method": "integer-floor-ceil-crop-adaptive-max-not-roi-align"}
SPLIT47 = {"kind": "controlled-three-image-optimization", "seed": SEED,
           "independent_test": False, "claim": "wiring-and-optimization-only"}
PREPROCESS47 = {"range": [0.0, 1.0], "dtype": "float32", "layout": "NCHW",
                "padding_mask": "bool-True-is-invalid-zero-before-conv", "fully_padded_empty_gt": "zero-loss"}
COORDINATES47 = {"format": "continuous-xyxy-exclusive-upper-bound", "clip": "[0,W]x[0,H]",
                 "width_height": "x2-x1,y2-y1", "roi_scale": "floor-left-top-ceil-right-bottom"}
ANCHORS_RECIPE47 = {"flatten_order": "y-x-ratio", "center": "(index+0.5)*image/feature",
                    "size": 8.0, "aspect_ratios": [0.5, 1.0, 2.0], "ratio_definition": "width/height",
                    "border_policy": "allow-straddling-then-clip-proposals"}
TRAINING_RECIPE47 = {
    "optimizer": "Adam", "lr": 3e-3, "steps": TRAIN_STEPS47,
    "rpn": {"positive_iou": RPN_POSITIVE_IOU47, "negative_iou": RPN_NEGATIVE_IOU47,
            "batch_size": RPN_BATCH_SIZE47, "positive_fraction": RPN_POSITIVE_FRACTION47,
            "smooth_l1_beta": RPN_SMOOTH_L1_BETA47,
            "classification_denominator": "sampled-anchor-count-or-differentiable-zero",
            "regression_denominator": "positive-anchor-coordinate-count"},
    "roi": {"positive_iou": ROI_POSITIVE_IOU47, "negative_iou": "below-positive",
            "ignore": None, "sampler": ROI_SAMPLER47, "smooth_l1_beta": ROI_SMOOTH_L1_BETA47,
            "classification_denominator": "proposal-count-per-image-then-image-mean",
            "regression_denominator": "positive-roi-coordinate-count-then-image-mean"},
}
INFERENCE_RECIPE47 = {
    "rpn_score": RPN_PROPOSAL_SCORE47, "rpn_pre_nms_topk": RPN_PRE_NMS_TOPK47,
    "rpn_post_nms": RPN_POST_NMS47, "rpn_nms_iou": RPN_NMS_IOU47, "min_box_size": RPN_MIN_SIZE47,
    "proposal_stop_gradient": True, "roi_score_default": ROI_SCORE_THRESHOLD47,
    "roi_per_class_nms_iou": ROI_NMS_IOU47, "box_regression": "class-agnostic",
}
LABELS47 = {"0": "background", "1": "class-one-square", "2": "class-two-square"}

state47 = {k: v.detach().cpu().clone() for k, v in detector47.state_dict().items()}
buffer47 = io.BytesIO(); torch.save(state47, buffer47)
artifact47 = {
    "artifact_id": "tiny-faster-rcnn-squares-v1", "config": CONFIG47,
    "state_hex": buffer47.getvalue().hex(), "state_digest": state_digest47(state47),
    "data_digest": data_digest47(train_images47, train_padding47, train_targets47),
    "split": SPLIT47, "preprocess": PREPROCESS47, "coordinates": COORDINATES47,
    "anchors": ANCHORS_RECIPE47, "training_recipe": TRAINING_RECIPE47,
    "inference_recipe": INFERENCE_RECIPE47, "labels": LABELS47,
}
artifact47["package_digest"] = package_digest47(artifact47)
PUBLISHER_REGISTRY47 = MappingProxyType({artifact47["artifact_id"]: artifact47["package_digest"]})

@dataclass(frozen=True)
class PublishedDetector:
    model: TinyFasterRCNN
    config: object
    preprocess: object
    labels: object
    coordinates: object
    anchors: object
    training: object
    inference: object

    def forward(self, images, padding_mask):
        if images.dtype != torch.float32 or not torch.isfinite(images).all():
            raise ValueError("published detector expects finite float32 images")
        if images.numel() and (float(images.min()) < 0 or float(images.max()) > 1):
            raise ValueError("published detector expects image range [0,1]")
        if padding_mask.dtype != torch.bool:
            raise ValueError("published detector expects bool padding mask")
        return self.model(images, padding_mask, targets=None,
                          score_threshold=float(self.inference["rpn_score"]))

def load_published_detector47(package):
    artifact_id = package.get("artifact_id")
    if artifact_id not in PUBLISHER_REGISTRY47 or package.get("package_digest") != PUBLISHER_REGISTRY47[artifact_id]:
        raise ValueError("artifact is not approved by publisher registry")
    if package_digest47(package) != package["package_digest"]:
        raise ValueError("package digest mismatch")
    expected = {"config": CONFIG47, "split": SPLIT47, "preprocess": PREPROCESS47,
                "coordinates": COORDINATES47, "anchors": ANCHORS_RECIPE47,
                "training_recipe": TRAINING_RECIPE47, "inference_recipe": INFERENCE_RECIPE47,
                "labels": LABELS47}
    for field, wanted in expected.items():
        if package.get(field) != wanted:
            raise ValueError(f"published detector contract mismatch: {field}")
    if set(package["labels"]) != {str(i) for i in range(package["config"]["num_classes"] + 1)}:
        raise ValueError("detector labels and num_classes disagree")
    if package.get("data_digest") != data_digest47(train_images47, train_padding47, train_targets47):
        raise ValueError("bound controlled data mismatch")
    state = torch.load(io.BytesIO(bytes.fromhex(package["state_hex"])), map_location="cpu", weights_only=True)
    if state_digest47(state) != package["state_digest"]:
        raise ValueError("canonical state digest mismatch")
    model = TinyFasterRCNN(num_classes=package["config"]["num_classes"]).eval()
    model.load_state_dict(state, strict=True)
    return PublishedDetector(model, deep_freeze47(package["config"]), deep_freeze47(package["preprocess"]),
                             deep_freeze47(package["labels"]), deep_freeze47(package["coordinates"]),
                             deep_freeze47(package["anchors"]), deep_freeze47(package["training_recipe"]),
                             deep_freeze47(package["inference_recipe"]))

loaded_detector47 = load_published_detector47(deepcopy(artifact47))
assert isinstance(loaded_detector47.model, TinyFasterRCNN)
assert loaded_detector47.labels["0"] == "background"
assert loaded_detector47.inference["proposal_stop_gradient"] is True
try:
    loaded_detector47.labels["1"] = "mutated"
    raise AssertionError("published detector labels should be read-only")
except TypeError:
    pass

forged47 = deepcopy(artifact47)
forged_state47 = torch.load(io.BytesIO(bytes.fromhex(forged47["state_hex"])), weights_only=True)
forged_state47["head.classifier.bias"] = forged_state47["head.classifier.bias"].roll(1)
forged_buffer47 = io.BytesIO(); torch.save(forged_state47, forged_buffer47)
forged47["state_hex"] = forged_buffer47.getvalue().hex()
forged47["state_digest"] = state_digest47(forged_state47)
forged47["labels"] = {"0": "background", "1": "forged", "2": "mapping"}
forged47["package_digest"] = package_digest47(forged47)
try:
    load_published_detector47(forged47)
    raise AssertionError("fully rehashed forged detector must fail closed")
except ValueError as exc:
    assert "publisher registry" in str(exc)


## 12. 失败模式、复杂度与生产差距

- **空集合**：空 GT、空 proposals、某类别无检测都是正常分支；对空 tensor 直接 `.mean()` 会产生 NaN。
- **坐标与 scale**：本例固定 32×32 和连续 `xyxy`；真实 resize/letterbox 必须保存原尺寸与 scale，推理后映射回原图。
- **ROI 算子**：整数 pooling 有量化误差，不可在文档或模型名中冒充 ROI Align。生产应使用经过数值/梯度验证的高性能算子。
- **padding 感受野**：本例保守清零所有接触 padding 的 feature；真实 FPN 多尺度需要逐层传播 mask。
- **复杂度**：RPN 是 $O(HWA)$；朴素 NMS 最坏 $O(P^2)$；逐 ROI Python 循环只适合教学。
- **发布语义**：`PublishedDetector` 携带只读预处理、坐标、anchor、训练分母与推理阈值；真实 registry 应由签名/KMS 托管。
- **评估**：三张训练图上的 loss 下降不代表泛化。生产需要独立 train/val/test、COCO 风格 AP、类别/尺度分桶、阈值校准和延迟压测。

原始资料：

- [Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks](https://arxiv.org/abs/1506.01497)
- [Fast R-CNN（ROI pooling）](https://arxiv.org/abs/1504.08083)
- [Mask R-CNN（ROI Align）](https://arxiv.org/abs/1703.06870)
